In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
print("start")
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install filterpy

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

# ĐỌC DỮ LIỆU

In [ ]:
img_root = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad/resized_images_only"
box_file = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad/all_bboxes.jsonl"

sequences = ['20240914_283b5d348658a68226783cdeab363a88_10min24s.frame', 
             '20240914_283b5d348658a68226783cdeab363a88_1min12s.frame',
             '20240914_283b5d348658a68226783cdeab363a88_6min09s.frame', 
             '20240914_283b5d348658a68226783cdeab363a88_6min47s.frame',
             '20240914_c2b43e749bd87dd58fbd3fc5a436171d_45s.frame',
            '20240918-youtube_short_00a696f1175f121794fb27c61c9db21c_1min21s.frame',
             '20240918-youtube_short_00a696f1175f121794fb27c61c9db21c_1min37s.frame',
             '20240918-youtube_short_00a696f1175f121794fb27c61c9db21c_1min46s.frame',
             '20240918-youtube_short_00a696f1175f121794fb27c61c9db21c_2min37s.frame',
             '20240914_283b5d348658a68226783cdeab363a88_8min50s.frame'
            ]
print(sequences)

In [ ]:
import json

all_boxes = []
with open(box_file, "r") as f:
    for line in f:
        all_boxes.append(json.loads(line))

In [ ]:
import os
from collections import defaultdict

def load_sequence(seq_name):
    seq_path = os.path.join(img_root, seq_name)

    # Load ảnh
    frames = sorted(
        [f for f in os.listdir(seq_path) if f.endswith(".jpg")],
        key=lambda x: int(x.split(".")[0])
    )

    assert len(frames) == 9, f"{seq_name} không có 9 frame!"

    images = [os.path.join(seq_path, f) for f in frames]

    # Lọc bbox của sequence đó
    seq_boxes = [b for b in all_boxes if b["folder_id"] == seq_name]

    # Gom theo frame_id
    boxes_by_frame = defaultdict(list)
    for b in seq_boxes:
        boxes_by_frame[b["frame_id"]].append(b)

    # Tạo list detection theo thứ tự frame
    detections = []
    for i in range(9):
        detections.append(boxes_by_frame.get(i, []))

    return images, detections


In [ ]:
for seq in sequences:
    images, dets = load_sequence(seq)
    print(seq, "→", len(images), "frames,", len(dets), "detection lists")

# HÀM HỖ TRỢ

In [ ]:
def convert_bbox(bbox):
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = bbox[0] + w/2.
    y = bbox[1] + h/2.
    s = w * h
    r = w / float(h) if h > 0 else 1.0
    return np.array([[x], [y], [s], [r]]).reshape((4, 1))

def get_box_coords(state):
    """
    Chuyển từ [x, y, s, r] về [x1, y1, x2, y2] để tính IOU
    """
    x, y, s, r = state
    w = np.sqrt(s * r)
    h = s / w
    x1 = x - w / 2.
    y1 = y - h / 2.
    x2 = x + w / 2.
    y2 = y + h / 2.
    return [x1, y1, x2, y2]

def calculate_iou(box1, box2):
    """
    box1, box2: [x1, y1, x2, y2]
    """
    xx1 = max(box1[0], box2[0])
    yy1 = max(box1[1], box2[1])
    xx2 = min(box1[2], box2[2])
    yy2 = min(box1[3], box2[3])

    w = max(0, xx2 - xx1)
    h = max(0, yy2 - yy1)
    inter = w * h
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def filter_small_objects(detections, img_w, img_h, min_ratio):
    """
    Lọc các detection có diện tích nhỏ hơn min_ratio * diện tích ảnh.
    """
    img_area = img_w * img_h
    min_area = img_area * min_ratio
    
    valid_detections = []
    
    for det in detections:
        box = det['boxs'] # [x1, y1, x2, y2]
        w = box[2] - box[0]
        h = box[3] - box[1]
        area = w * h
        
        # Chỉ giữ lại nếu diện tích đủ lớn
        if area >= min_area:
            valid_detections.append(det)
            
    return valid_detections

In [ ]:
def get_motion_info(trk):
    """Tính toán vận tốc và xác định hướng di chuyển"""
    vx = trk.kf.x[4][0]
    vy = trk.kf.x[5][0]
    speed = np.sqrt(vx**2 + vy**2)
    angle = np.degrees(np.arctan2(vy, vx))
    direction_str = "---"
    if speed > 5: # Ngưỡng tốc độ
        if abs(vx) > abs(vy):
            direction_str = "Sang PHAI" if vx > 0 else "Sang TRAI"
        else:
            # y tăng là đi xuống (gần), y giảm là đi lên (xa)
            direction_str = "LAI GAN (Nguy hiem)" if vy > 0 else "DI XA"
    else:
        direction_str = "Dung yen / Cham"
        
    return speed, vx, vy, angle, direction_str

In [ ]:
def angle_difference(angle1, angle2):
    """Tính độ lệch nhỏ nhất giữa hai góc (trả về giá trị 0-180)"""
    diff = abs(angle1 - angle2) % 360
    if diff > 180:
        diff = 360 - diff
    return diff

In [ ]:
def create_json_entry(folder_id, frame_id, det_info, tracker=None):
    entry = det_info.copy()
    entry["folder_id"] = str(folder_id)
    entry["frame_id"] = int(frame_id)

    if tracker is not None:
        vx, vy = getattr(tracker, "velocity_smooth", (0.0, 0.0))
        speed, vx, vy, kf_angle, direct = get_motion_info(tracker)
        # Lấy thông tin góc nối tâm (vừa thêm)
        cent_angle = getattr(tracker, "centroid_angle", 0.0)
        # Tính độ lệch giữa 2 phương pháp
        diff_angle = angle_difference(kf_angle, cent_angle)

        entry["tracking_info"] = {
            "track_id": int(tracker.id),
            "velocity_vector": [round(float(vx), 2), round(float(vy), 2)],
            "speed_px": round(float(speed), 2),
            # "angle_deg": round(float(angle), 1),
            "kf_angle": round(float(kf_angle), 1),       # Góc Kalman
            "cent_angle": round(float(cent_angle), 1),   # Góc nối tâm
            "diff_angle": round(float(diff_angle), 1),   # Độ lệch
            # "direction": direct
        }
    else:
        entry["tracking_info"] = {
            "track_id": -1,
            "velocity_vector": [0.0, 0.0],
            "speed_px": 0.0,
            # "angle_deg": 0.0
            "kf_angle": 0.0,
            "cent_angle": 0.0,
            "diff_angle": 0.0,
            # "direction": "New/Unknown"
        }

    return entry


In [ ]:
# def print_tabular_report(frame_idx, json_results):
#     print(f"\n>>> BÁO CÁO FRAME {frame_idx}")
# # Header bảng: Thêm cột Cent.Ang (Centroid) và Diff (Độ lệch)
#     print(f"{'ID':<4} | {'Label':<10} | {'Vận tốc (vx, vy)':<18} | {'Speed':<6} | {'KF.Ang':<7} | {'Cent.Ang':<8} | {'Diff':<5} | {'HƯỚNG'}")
#     print("-" * 100)
    
#     for item in json_results:
#         info = item["tracking_info"]
#         vx, vy = info["velocity_vector"]
#         print(f"{info['track_id']:<4} | {item['label']:<10} | ({vx:>6.1f}, {vy:>6.1f})    | {info['speed_px']:<6.1f} | {info['angle_deg']:<6.0f} | {info['direction']}")
#     print("=" * 80)
def print_tabular_report(frame_idx, json_results):
    print(f"\n>>> BÁO CÁO FRAME {frame_idx}")
    # Header bảng: Thêm cột Cent.Ang (Centroid) và Diff (Độ lệch)
    print(f"{'ID':<4} | {'Label':<10} | {'Vận tốc (vx, vy)':<18} | {'Speed':<6} | {'KF.Ang':<7} | {'Cent.Ang':<8} | {'Diff':<5} | {'HƯỚNG'}")
    print("-" * 100)
    
    for item in json_results:
        info = item["tracking_info"]
        vx, vy = info["velocity_vector"]
        
        # Lấy dữ liệu mới
        kf_ang = info.get('kf_angle', 0)
        cent_ang = info.get('cent_angle', 0)
        diff = info.get('diff_angle', 0)
        
        print(f"{info['track_id']:<4} | {item['label']:<10} | ({vx:>6.1f}, {vy:>6.1f})    | {info['speed_px']:<6.1f} | {kf_ang:<7.0f} | {cent_ang:<8.0f} | {diff:<5.0f} ")
    print("=" * 100)

# Kalman Prediction + Association + Kalman Update¶

In [ ]:
class Tracker:
    def __init__(self, track_id, initial_box, label):
        self.id = track_id
        self.label = label
        self.time_since_update = 0
        self.hits = 0 
        self.velocity_smooth = np.array([0.0, 0.0]) 

        # --- MỚI: Biến lưu góc nối tâm ---
        self.centroid_angle = 0.0
        # ---------------------------------

        box = convert_bbox(initial_box) 
        self.last_pos = np.array([float(box[0].item()), float(box[1].item())]) 

        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        # ... (Phần khởi tạo ma trận giữ nguyên như code của bạn) ...
        dt = 1 
        self.kf.F = np.array([
            [1, 0, 0, 0, dt, 0, 0], 
            [0, 1, 0, 0, 0, dt, 0], 
            [0, 0, 1, 0, 0, 0, dt],
            [0, 0, 0, 1, 0, 0, 0],  
            [0, 0, 0, 0, 1, 0, 0],  
            [0, 0, 0, 0, 0, 1, 0], 
            [0, 0, 0, 0, 0, 0, 1]
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0], 
            [0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 1, 0, 0, 0, 0], 
            [0, 0, 0, 1, 0, 0, 0]
        ])
        self.kf.R[2:, 2:] *= 10.  
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        # self.kf.Q[4:, 4:] *= 0.01
        self.kf.Q[4:, 4:] *= 0.099  # mới sửa để update góc theo quan sát
        self.kf.x[:4] = convert_bbox(initial_box)

        self.last_raw_det = None

    def apply_camera_motion(self, M):
        # ... (Giữ nguyên như code của bạn) ...
        cx = float(self.kf.x[0])
        cy = float(self.kf.x[1])
        pos = np.array([cx, cy, 1.0]) 
        new_pos = M @ pos 
        self.kf.x[0] = new_pos[0]
        self.kf.x[1] = new_pos[1]
        
        last_pos_vec = np.array([self.last_pos[0], self.last_pos[1], 1.0])
        new_last_pos = M @ last_pos_vec
        
        self.last_pos[0] = new_last_pos[0]
        self.last_pos[1] = new_last_pos[1]

    def predict(self):
        # ... (Giữ nguyên như code của bạn) ...
        if (float(self.kf.x[6]) + float(self.kf.x[2])) <= 0: 
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1

    def update(self, bbox, raw_det=None):
        self.time_since_update = 0
        self.hits += 1      
        
        # 1. Cập nhật Kalman Filter để có vị trí mới nhất (đã lọc nhiễu)
        self.kf.update(convert_bbox(bbox)) 

        # --- PHẦN CODE BỔ SUNG TÍNH GÓC NỐI TÂM ---
        # Lấy vị trí hiện tại (Current Position)
        curr_x = float(self.kf.x[0])
        curr_y = float(self.kf.x[1])

        # Tính vector di chuyển: P_hientai - P_cu (đã bù camera motion ở hàm apply_camera_motion)
        dx = curr_x - self.last_pos[0]
        dy = curr_y - self.last_pos[1]

        # Tính góc (độ). Thêm ngưỡng (ví dụ > 0.5 pixel) để tránh nhiễu khi vật đứng yên
        if (dx**2 + dy**2) > 2.0:
            self.centroid_angle = np.degrees(np.arctan2(dy, dx))
        else:
            # Nếu đứng yên, gán bằng hướng của Kalman (để độ lệch = 0)
            kvx = float(self.kf.x[4]) # Vận tốc x từ Kalman
            kvy = float(self.kf.x[5]) # Vận tốc y từ Kalman
            self.centroid_angle = np.degrees(np.arctan2(kvy, kvx))
        # -------------------------------------------

        # --- PHẦN 2: HIỂN THỊ VẬN TỐC & HƯỚNG (Code cũ của bạn) ---
        raw_vx = float(self.kf.x[4])
        raw_vy = float(self.kf.x[5])
        
        alpha = 0.15
        if self.hits <= 2:
            self.velocity_smooth = np.array([raw_vx, raw_vy])
        else:
            self.velocity_smooth = alpha * np.array([raw_vx, raw_vy]) + \
                                   (1 - alpha) * self.velocity_smooth
        
        # Cập nhật last_pos để dùng cho frame kế tiếp
        # LƯU Ý: Dòng này bắt buộc phải nằm SAU bước tính dx, dy
        self.last_pos = np.array([curr_x, curr_y])
        
        if raw_det is not None:
            self.last_raw_det = raw_det


In [ ]:
def associate_object(detections, trackers):
    if len(trackers) == 0:
        return np.empty((0, 2), dtype=int), list(range(len(detections))), []

    det_rects = [det['boxs'] for det in detections]
    det_states = [convert_bbox(d['boxs']).flatten() for d in detections]

    cost_matrix = np.zeros((len(trackers), len(detections)))

    for t, trk in enumerate(trackers):
        trk_state = trk.kf.x.flatten()[:4]   # [x, y, s, r]
        trk_rect = get_box_coords(trk_state)
        # convert tracker state → w, h
        s_t, r_t = trk_state[2], trk_state[3]
        w_trk = np.sqrt(s_t * r_t)
        h_trk = s_t / w_trk

        for d, det in enumerate(detections):
            if trk.label != det['label']:
                cost_matrix[t, d] = 1e9
                continue

            det_state = det_states[d]   # [x, y, s, r]
            det_rect = det_rects[d]

            # convert detection state → w, h
            s_d, r_d = det_state[2], det_state[3]
            w_det = np.sqrt(s_d * r_d)
            h_det = s_d / w_det
            if h_det <= 0:
                h_det = 1.0

            # center distance
            dist = np.linalg.norm(trk_state[:2] - det_state[:2])
            norm_dist = dist / h_det

            # relative scale change (height)
            shape_diff = abs(h_trk - h_det) / h_det

            # IOU
            iou = calculate_iou(trk_rect, det_rect)
            iou_cost = 1.0 - iou

            cost_matrix[t, d] = 0.5*norm_dist + 0.2*shape_diff + 0.2* iou_cost
            
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches = []
    matched_trackers = set()
    matched_detections = set()
    threshold = 4.5

    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] <= threshold:
            matches.append([r, c])
            matched_trackers.add(r)
            matched_detections.add(c)

    unmatched_dets = list(set(range(len(detections))) - matched_detections)
    unmatched_trks = list(set(range(len(trackers))) - matched_trackers)

    matches = np.array(matches) if len(matches) > 0 else np.empty((0, 2), dtype=int)

    return matches, unmatched_dets, unmatched_trks

In [ ]:
class CameraMotionEstimator:
    def __init__(self):
        self.detector = cv2.ORB_create(nfeatures=500)
        self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        self.prev_kp = None
        self.prev_des = None
        
    def estimate(self, curr_frame):
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        curr_kp, curr_des = self.detector.detectAndCompute(curr_gray, None)

        if self.prev_kp is None or curr_des is None or len(curr_kp) < 20:
            self.prev_kp = curr_kp
            self.prev_des = curr_des
            return np.eye(3)

        matches = self.matcher.match(self.prev_des, curr_des)
        src_pts = []
        dst_pts = []
        for m in matches:
            src_pts.append(self.prev_kp[m.queryIdx].pt)
            dst_pts.append(curr_kp[m.trainIdx].pt)

        src_pts = np.float32(src_pts).reshape(-1, 1, 2)
        dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)

        if len(matches) > 10:
            M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC)
            if M is None: 
                M_3x3 = np.eye(3)
            else:
                M_3x3 = np.eye(3)
                M_3x3[:2] = M
        else:
            M_3x3 = np.eye(3)

        self.prev_kp = curr_kp
        self.prev_des = curr_des
        return M_3x3

# LOOP mới

In [ ]:
# ============================================================
# 3. VÒNG LẶP CHÍNH (MAIN LOOP) - ĐÃ CẬP NHẬT
# ============================================================
import math

# Chỉ chạy thử sequence [3:4]
sequences_test = sequences[8:9] 
final_json_output = [] 

for seq in sequences_test:
    print(f"\n{'='*60}\nProcessing Sequence: {seq}\n{'='*60}")
    
    image_paths, seq_detections = load_sequence(seq)
    
    if not image_paths:
        print("Skipping (no images)...")
        continue

    motion_estimator = CameraMotionEstimator()
    tracker_list = []
    track_id_count = 1 
    
    for k in range(len(image_paths)):
        img_path = image_paths[k]
        frame_dets = seq_detections[k] 
        
        img = cv2.imread(img_path)
        if img is None: continue
        
        # --- A. Tracking Core ---
        M = motion_estimator.estimate(img)
        
        for trk in tracker_list: 
            trk.predict()               
            trk.apply_camera_motion(M) 
            
        matches, unmatched_dets, unmatched_trks = associate_object(frame_dets, tracker_list)
        
        frame_report_list = [] 
        
        # --- B. Cập nhật Tracker đã match ---
        for trk_idx, det_idx in matches:
            trk = tracker_list[trk_idx]
            det_info = frame_dets[det_idx]
            
            # Cập nhật Tracker (Logic tính góc nối tâm nằm trong hàm update mới)
            trk.update(det_info['boxs'], raw_det=det_info)
            
            # TASK 3: Tạo JSON record (Hàm create_json_entry đã sửa để lưu thêm góc)
            json_obj = create_json_entry(seq, k, det_info, trk)
            final_json_output.append(json_obj)
            frame_report_list.append(json_obj)
            
            # TASK 1: Visualization (Vẽ LABEL + ANGLE + DIFF)
            x1, y1, x2, y2 = [int(v) for v in det_info['boxs']]
            color = (0, 255, 0) if trk.hits >= 2 else (0, 255, 255)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
            
            # Lấy thông tin vận tốc & góc
            speed, vx, vy, kf_angle, direct = get_motion_info(trk)
            cent_angle = getattr(trk, "centroid_angle", 0.0)
            
            # Vẽ Label text: Hiển thị cả KF Angle (K) và Centroid Angle (C)
            # K: Góc Kalman, C: Góc Nối tâm
            label_text = f"ID:{trk.id} K:{int(kf_angle)} C:{int(cent_angle)}"
            cv2.putText(img, label_text, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            
            # Vẽ mũi tên Kalman (Màu ĐỎ)
            if speed > 2.0: 
                cx, cy = int((x1+x2)/2), int((y1+y2)/2)
                end_x, end_y = int(cx + vx), int(cy + vy) 
                cv2.arrowedLine(img, (cx, cy), (end_x, end_y), (0, 0, 255), 2, tipLength=0.3)
                
                # (Tùy chọn) Vẽ mũi tên Centroid (Màu XANH DƯƠNG) để so sánh trực quan
                # Tính vx, vy giả lập từ góc centroid để vẽ
                c_rad = math.radians(cent_angle)
                cvx = speed * math.cos(c_rad)
                cvy = speed * math.sin(c_rad)
                end_cx, end_cy = int(cx + cvx), int(cy + cvy)
                cv2.arrowedLine(img, (cx, cy), (end_cx, end_cy), (255, 0, 0), 2, tipLength=0.3)

        # --- C. Tạo Tracker mới ---
        for det_idx in unmatched_dets:
            det_info = frame_dets[det_idx]
            new_trk = Tracker(track_id_count, det_info['boxs'], det_info['label'])
            tracker_list.append(new_trk)
            track_id_count += 1
            
            json_obj = create_json_entry(seq, k, det_info, new_trk)
            final_json_output.append(json_obj)
            frame_report_list.append(json_obj)
            
            x1, y1, x2, y2 = [int(v) for v in det_info['boxs']]
            cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 1)

        # Lọc tracker cũ
        tracker_list = [t for t in tracker_list if t.time_since_update <= 2]

        # --- TASK 2: In Báo Cáo (Cập nhật bảng report) ---
        # Sử dụng hàm print_tabular_report mới thay vì print thủ công
        print_tabular_report(k, frame_report_list)

        # --- TASK 1: Visualize ---
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 10))
        plt.imshow(img_rgb)
        plt.title(f"{seq} | Frame {k}")
        plt.axis('off')
        plt.show()

# ============================================================
# KẾT QUẢ CUỐI CÙNG
# ============================================================
print(f"\n>>> PROCESSING COMPLETE.")
print(f"Total Enriched Objects Generated: {len(final_json_output)}")

# --- LỌC VÀ IN RA 5 MẪU CỦA FRAME 8 ---
print("\n>>> SAMPLES FROM FRAME 8 (CHECK VELOCITY & ANGLES HERE):")
frame_8_samples = [obj for obj in final_json_output if obj['frame_id'] == 8]
print(json.dumps(frame_8_samples[:5], indent=2))

# TEST

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import cv2 

# ==============================================================================
# 1. CẤU HÌNH CHẠY TOÀN BỘ DỮ LIỆU
# ==============================================================================
run_sequences = run_sequences[:20] 
# run_sequences = sequences[:10] # Chạy thử 10 cái đầu nếu muốn test nhanh

print(f"Đang tiến hành Tracking và Thống kê trên {len(run_sequences)} sequences...")

# List lưu trữ toàn bộ giá trị Diff Angle để thống kê
all_diff_angles = []

# ==============================================================================
# 2. VÒNG LẶP TRACKING CHÍNH 
# ==============================================================================
for seq_idx, seq in enumerate(run_sequences):
    if seq_idx % 10 == 0:
        print(f"Processing {seq_idx}/{len(run_sequences)}: {seq}")
        
    try:
        image_paths, seq_detections = load_sequence(seq)
    except Exception as e:
        continue
    
    if not image_paths: continue

    motion_estimator = CameraMotionEstimator()
    tracker_list = []
    track_id_count = 1 
    
    for i in range(len(image_paths)):
        frame_idx = i
        raw_dets = seq_detections[i] if i < len(seq_detections) else []
        img = cv2.imread(image_paths[i])
        if img is None: continue 

        # 1. Update Camera Motion (Đã sửa lỗi gọi hàm)
        try:
            M = motion_estimator.estimate(img)
        except AttributeError:
            M = np.eye(3) 
        
        # 2. Apply Motion Compensation & Predict
        for trk in tracker_list:
            if M is not None:
                trk.apply_camera_motion(M)
            trk.predict()
            
        # 3. Associate & Update
        matches, unmatched_dets, unmatched_trks = associate_object(raw_dets, tracker_list)
        
        for t_idx, d_idx in matches:
            trk = tracker_list[t_idx]
            det = raw_dets[d_idx]
            trk.update(det["boxs"], raw_det=det)
            
            # --- [ĐOẠN CODE SỬA LỖI TYPE ERROR] ---
            if 3 <= frame_idx <= 8:
                if hasattr(trk, 'kf') and hasattr(trk, 'centroid_angle'):
                    # Ép kiểu float để tránh lỗi numpy array
                    vx = float(trk.kf.x[4])
                    vy = float(trk.kf.x[5])
                    speed = np.sqrt(vx**2 + vy**2)
                    
                    if speed > 2.0:
                        kf_angle = np.degrees(np.arctan2(vy, vx))
                        cent_angle = float(trk.centroid_angle)
                        
                        diff = abs(kf_angle - cent_angle) % 360
                        if diff > 180: diff = 360 - diff
                        
                        # Quan trọng: float(diff)
                        all_diff_angles.append(float(diff))
            # --------------------------------------

        # Tạo tracker mới
        for d_idx in unmatched_dets:
            det = raw_dets[d_idx]
            new_trk = Tracker(track_id_count, det["boxs"], det["label"])
            new_trk.last_raw_det = det
            tracker_list.append(new_trk)
            track_id_count += 1
            
        tracker_list = [t for t in tracker_list if t.time_since_update < 3]

# ==============================================================================
# 3. VẼ BIỂU ĐỒ VÀ BẢNG THỐNG KÊ
# ==============================================================================
if len(all_diff_angles) > 0:
    # Code vẽ biểu đồ giữ nguyên
    diff_series = pd.Series(all_diff_angles)
    
    total_samples = len(diff_series)
    acceptable = diff_series[diff_series < 10].count()
    unacceptable = total_samples - acceptable
    acc_rate = (acceptable / total_samples) * 100
    
    print("\n" + "="*50)
    print(" KẾT QUẢ THỐNG KÊ ĐỘ LỆCH GÓC (DIFF ANGLE)")
    print("="*50)
    print(f"Tổng số mẫu (Moving Objects): {total_samples}")
    print(f"Trung bình (Mean):            {diff_series.mean():.2f}°")
    print(f"Trung vị (Median):            {diff_series.median():.2f}°")
    print("-" * 50)
    print(f"✅ CHẤP NHẬN ĐƯỢC (< 20°):     {acceptable} ({acc_rate:.1f}%)")
    print(f"❌ SAI KHÁC LỚN (>= 20°):      {unacceptable} ({100-acc_rate:.1f}%)")
    print("="*50)

    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    sns.histplot(all_diff_angles, bins=36, kde=True, color='skyblue')
    plt.axvline(x=20, color='r', linestyle='--', label='Ngưỡng 20°')
    plt.title(f'Phân phối độ lệch (Mean: {diff_series.mean():.1f}°)')
    plt.xlabel('Độ lệch (Độ)')
    plt.legend()

    plt.subplot(1, 2, 2)
    labels = [f'Acceptable (<20°) - {acc_rate:.1f}%', f'Large Diff (>=20°) - {100-acc_rate:.1f}%']
    plt.pie([acceptable, unacceptable], explode=(0.1, 0), labels=labels, colors=['#66b3ff', '#ff9999'], autopct='%1.1f%%', shadow=True, startangle=90)
    plt.title('Tỷ lệ mẫu đạt chuẩn')
    plt.axis('equal')  
    plt.show()

else:
    print("Không thu thập được dữ liệu nào! (Có thể do không có object nào di chuyển > 2px)")

In [ ]:
import os
import random

# 1. Cấu hình đường dẫn
img_root = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad/resized_images_only"

# 2. TỰ ĐỘNG lấy danh sách folder
# Lấy tất cả folder trong đường dẫn img_root
all_folders = [f for f in os.listdir(img_root) if os.path.isdir(os.path.join(img_root, f))]

# Sắp xếp để thứ tự chạy nhất quán (hoặc random nếu muốn test ngẫu nhiên)
all_folders.sort() 
# random.shuffle(all_folders) # Bỏ comment dòng này nếu muốn lấy 1000 mẫu ngẫu nhiên

# 3. Chọn số lượng mẫu muốn test
NUM_SAMPLES = 1000

# Lấy 1000 mẫu đầu tiên (hoặc toàn bộ nếu dataset nhỏ hơn 1000)
run_sequences = all_folders[:NUM_SAMPLES]

print(f"✅ Đã tìm thấy tổng cộng: {len(all_folders)} sequences.")
print(f"🚀 Đang thiết lập chạy test trên: {len(run_sequences)} sequences.")
print("5 sequences đầu tiên:", run_sequences[:5])